In [2]:

import os
import argparse
import tensorflow as tf
import numpy as np
import sys
from scipy.io import loadmat, savemat


In [ ]:

# Create argument parser
parser = argparse.ArgumentParser(
    description='Predict polarity from .mat file waveforms using CFM model'
)
parser.add_argument(
    '--wdir',
    type=str,
    default='/Users/mczhang/Documents/GitHub/FM5_ML/',
    help='Base directory containing 02-data/A_wave/A_wave_dB15.mat'
)
parser.add_argument(
    '--model',
    type=str,
    default='./Network_CFM/CFM_with_timeshift.hdf5',
    help='Path to the CFM model file'
)

# Filter out Jupyter-specific arguments
valid_args = [arg for arg in sys.argv if not arg.startswith('--f=')]

# Parse arguments
args = parser.parse_args(valid_args[1:])  # Skip the first argument (script name or kernel launcher)

# File paths
mat_file_path = os.path.join(args.wdir, '02-data', 'F_ML', 'A_wave_dB15_DT_test.mat')
output_mat_file = os.path.join(args.wdir, '02-data', 'F_ML', 'A_wave_dB15_DT_CFM.mat')
model_path = args.model

# Verify input files
if not os.path.isfile(mat_file_path):
    print(f"Error: .mat file not found at {mat_file_path}")
    sys.exit(1)
if not os.path.isfile(model_path):
    print(f"Error: Model file not found at {model_path}")
    sys.exit(1)

print(f"Input .mat file: {mat_file_path}")
print(f"Output .mat file: {output_mat_file}")
print(f"Model file: {model_path}")

Input .mat file: /Users/mczhang/Documents/GitHub/FM5_ML/02-data/F_ML/A_wave_dB15_DT_test.mat
Output .mat file: /Users/mczhang/Documents/GitHub/FM5_ML/02-data/F_ML/A_wave_dB15_DT_CFM.mat
Model file: ./Network_CFM/CFM_with_timeshift.hdf5


In [8]:
# Load CFM model
try:
    cfm_model = tf.keras.models.load_model(model_path, compile=False)
except Exception as e:
    print(f"Error loading model from {model_path}: {e}")
    sys.exit(1)

# Load Felix data
try:
    mat_data = loadmat(mat_file_path, struct_as_record=False, squeeze_me=True)
    Felix = mat_data['Felix']

    # Standardize to array
    if isinstance(Felix, np.ndarray):
        felix_data = Felix
        num_events = len(felix_data)
    else:
        felix_data = np.array([Felix])  # Convert single struct to 1-element array
        num_events = 1

    print(f"Found {num_events} events in the file")
except Exception as e:
    print(f"Error reading .mat file {mat_file_path}: {e}")
    sys.exit(1)

# Stations to process
waveform_fields = ['W_AS1', 'W_AS2', 'W_CC1', 'W_EC1', 'W_EC2', 'W_EC3', 'W_ID1']

# Initialize arrays for results
cfm_results = {field: np.zeros(num_events, dtype=np.int8) for field in waveform_fields}


Found 200 events in the file


2025-09-10 15:54:40.758166: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [10]:
# Process each event
counter = 0
for i in range(num_events):
    try:
        for field in waveform_fields:
            if not hasattr(felix_data[i], field):
                continue

            waveform = getattr(felix_data[i], field)
            waveform = np.array(waveform).flatten()

            if len(waveform) < 181:  # Need at least 180 samples
                continue

            # Extract 160 samples (20–180)
            wf_input = waveform[20:180]

            if np.max(np.abs(wf_input)) == 0:
                continue

            # Demean + normalize
            wf_input = wf_input - np.mean(wf_input)
            wf_input = wf_input / (np.max(np.abs(wf_input)) + 1e-6)

            # Reshape to (1, 160) for model
            wf_input = wf_input.reshape(1, -1)

            # Predict polarity
            pred = cfm_model.predict(wf_input, verbose=0)
            pred_val = pred.flatten()[0]

            if pred_val >= 0.5:
                cfm_results[field][i] = 1   # Upward
            else:
                cfm_results[field][i] = -1  # Downward

        counter += 1
        if counter % 100 == 0:
            print(f"Processed {counter} events")

    except Exception as e:
        print(f"Error processing event {i+1}: {e}")
        continue


Processed 100 events
Processed 200 events


In [27]:
cfm_results


{'W_AS1': array([ 0,  0,  0, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0, -1,  0,  0, -1,  0,  0,  0,  0,  0,  0,  0, -1,  0,
         0,  0, -1,  0,  0,  0,  0,  0, -1,  0,  0, -1,  0,  0,  0, -1,  0,
         0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0, -1,  0,  0,
         0,  0,  1,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], dtype=int8),
 'W_AS2': array([ 1,  1, -1, -1, -1,  0,  0,  0, -1,  1,  1, -1,  0, -1,  0,  0, 

In [19]:
print("Type of felix_data:", type(felix_data))
print("felix_data dtype:", felix_data.dtype)
print("felix_data shape:", felix_data.shape)
print("felix_data.dtype.names:", felix_data.dtype.names)

Type of felix_data: <class 'numpy.ndarray'>
felix_data dtype: object
felix_data shape: (200,)
felix_data.dtype.names: None


In [28]:
# Save results back to .mat
try:
    # Get field names from the first element
    if len(felix_data) == 0:
        raise ValueError("felix_data is empty")
    field_names = list(felix_data[0]._fieldnames)

    # Create new dtype
    new_dtype = []
    for field_name in field_names:
        field_value = getattr(felix_data[0], field_name)
        if isinstance(field_value, np.ndarray):
            field_dtype = field_value.dtype
            field_shape = field_value.shape
            new_dtype.append((field_name, field_dtype, field_shape))
        else:
            field_dtype = np.dtype(type(field_value))
            new_dtype.append((field_name, field_dtype))
    for field in waveform_fields:
        new_dtype.append((f'PoMLCFM_{field}', np.int8))

    # Create new structured array
    new_felix = np.zeros(num_events, dtype=new_dtype)

    # Copy original fields event-by-event to avoid stacking issues
    for field_name in field_names:
        for i in range(num_events):
            value = getattr(felix_data[i], field_name)
            # Ensure value matches expected shape
            if isinstance(value, np.ndarray) and value.shape != new_felix[field_name][i].shape:
                raise ValueError(f"Shape mismatch for field {field_name} in event {i}: expected {new_felix[field_name][i].shape}, got {value.shape}")
            new_felix[field_name][i] = value

    # Add predictions
    for field in waveform_fields:
        new_felix[f'PoMLCFM_{field}'] = cfm_results[field]

    # Update mat_data
    mat_data['Felix'] = new_felix

    # Debug: Check structure
    print("New Felix dtype:", new_felix.dtype)
    print("New Felix shape:", new_felix.shape)

    # Save file
    savemat(output_mat_file, mat_data, long_field_names=True)
    print(f"Results saved to {output_mat_file}")

except Exception as e:
    print(f"Error saving .mat file {output_mat_file}: {e}")
    sys.exit(1)

Error saving .mat file /Users/mczhang/Documents/GitHub/FM5_ML/02-data/F_ML/A_wave_dB15_DT_CFM.mat: Shape mismatch for field DDt_AS1 in event 102: expected (), got (0,)


AttributeError: 'tuple' object has no attribute 'tb_frame'